In [1]:
import re

import numpy as np
import pandas as pd
# import dask.dataframe as pd
import sqlite3

from sklearn.preprocessing import StandardScaler

#import xgboost

con = sqlite3.connect('../Dengue20X_timeseries_CPA_NoiseReduction.db')

cursor = con.cursor()
cursor.execute('SELECT name FROM sqlite_master WHERE type="table";')
print(cursor.fetchall())

KeyboardInterrupt: 

In [2]:
meta = pd.read_sql_query('SELECT ImageNumber, Image_Metadata_WellID, Image_Metadata_PlateID from MyExpt_Per_Image', con)
meta.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,TimeSeries_20221028_164132
1,2,A01,TimeSeries_20221028_164132
2,3,A01,TimeSeries_20221028_164132
3,4,A01,TimeSeries_20221028_164132
4,5,A01,TimeSeries_20221028_164132


In [10]:
def _letter_range(start, stop="{", step=1):
    """Yield a range of lowercase letters.""" 
    for ord_ in range(ord(start.upper()), ord(stop.upper()), step):
        yield chr(ord_)

PC, NC = [],[]
for l in list(_letter_range('A', 'Q')):
    for i in range(1, 5):
        NC.append(f"{l}{str(i).zfill(2)}")
    for j in range(5, 9):
        PC.append(f"{l}{str(j).zfill(2)}")

print(NC)
print()
print(PC)

['A01', 'A02', 'A03', 'A04', 'B01', 'B02', 'B03', 'B04', 'C01', 'C02', 'C03', 'C04', 'D01', 'D02', 'D03', 'D04', 'E01', 'E02', 'E03', 'E04', 'F01', 'F02', 'F03', 'F04', 'G01', 'G02', 'G03', 'G04', 'H01', 'H02', 'H03', 'H04', 'I01', 'I02', 'I03', 'I04', 'J01', 'J02', 'J03', 'J04', 'K01', 'K02', 'K03', 'K04', 'L01', 'L02', 'L03', 'L04', 'M01', 'M02', 'M03', 'M04', 'N01', 'N02', 'N03', 'N04', 'O01', 'O02', 'O03', 'O04', 'P01', 'P02', 'P03', 'P04']

['A05', 'A06', 'A07', 'A08', 'B05', 'B06', 'B07', 'B08', 'C05', 'C06', 'C07', 'C08', 'D05', 'D06', 'D07', 'D08', 'E05', 'E06', 'E07', 'E08', 'F05', 'F06', 'F07', 'F08', 'G05', 'G06', 'G07', 'G08', 'H05', 'H06', 'H07', 'H08', 'I05', 'I06', 'I07', 'I08', 'J05', 'J06', 'J07', 'J08', 'K05', 'K06', 'K07', 'K08', 'L05', 'L06', 'L07', 'L08', 'M05', 'M06', 'M07', 'M08', 'N05', 'N06', 'N07', 'N08', 'O05', 'O06', 'O07', 'O08', 'P05', 'P06', 'P07', 'P08']


In [11]:
highCount = pd.read_csv('WellswHighCellCount.csv')
highCountList = highCount['Image_Metadata_WellID'].unique().tolist()
highCountList

['A01',
 'A02',
 'A03',
 'B01',
 'B02',
 'B03',
 'B04',
 'C01',
 'C02',
 'C03',
 'C04',
 'D01',
 'D02',
 'D03',
 'D04',
 'E01',
 'E02',
 'E03',
 'E04',
 'F01',
 'F02',
 'F03',
 'F04',
 'G01',
 'G04',
 'H01',
 'H03',
 'H04',
 'I04',
 'J01',
 'J02',
 'K01',
 'L03',
 'L04',
 'M01',
 'M02',
 'M04',
 'N01',
 'N02',
 'N04',
 'O01',
 'O04',
 'P01',
 'P03',
 'P04',
 'A21',
 'A22',
 'A23',
 'A24',
 'B21',
 'B22',
 'B23',
 'C22',
 'C23',
 'D21',
 'D22',
 'D23',
 'D24',
 'E21',
 'E22',
 'E23',
 'E24',
 'F21',
 'F23',
 'G21',
 'G22',
 'G23',
 'G24',
 'H21',
 'H22',
 'H23',
 'I21',
 'I22',
 'I23',
 'I24',
 'J21',
 'J22',
 'J23',
 'J24',
 'K21',
 'K22',
 'K23',
 'K24',
 'L21',
 'L22',
 'L23',
 'L24',
 'M21',
 'M22',
 'M23',
 'M24',
 'N21',
 'N22',
 'N23',
 'N24',
 'O21',
 'O22',
 'O23',
 'A17',
 'A19',
 'B17',
 'B19',
 'C17',
 'G17',
 'G19',
 'H20',
 'I17',
 'I18',
 'I19',
 'I20',
 'J18',
 'K17',
 'K18',
 'K20',
 'L18',
 'L19',
 'M17',
 'M20',
 'N17',
 'O18',
 'O19',
 'O20',
 'P17',
 'P20',
 'A14',


In [12]:
#select only PC and NC with high cell count to train
PC_highCC = set(PC).intersection(set(highCountList))
NC_highCC = set(NC).intersection(set(highCountList))
print(PC_highCC)
print()
print(NC_highCC)

{'E08', 'M05', 'N07', 'F05', 'O05', 'G08', 'E06', 'E05', 'G05', 'B06', 'D06', 'C06', 'I06', 'L07', 'P05', 'J05', 'N08', 'C05', 'J07', 'A06', 'D05', 'H07', 'B07', 'P06', 'O08', 'B05', 'A05', 'I05', 'L06', 'J06', 'F06', 'O06', 'N05', 'K05', 'C07', 'L05', 'G06', 'H08', 'I08', 'H06', 'H05', 'I07', 'A08', 'M07', 'N06'}

{'F04', 'D04', 'O04', 'B04', 'M02', 'H04', 'L03', 'M01', 'A02', 'A03', 'N01', 'J01', 'E04', 'D03', 'E01', 'L04', 'I04', 'B02', 'C04', 'N02', 'F01', 'C02', 'C03', 'H01', 'B01', 'E03', 'P01', 'A01', 'D02', 'H03', 'D01', 'G01', 'E02', 'C01', 'O01', 'F03', 'K01', 'B03', 'G04', 'J02', 'N04', 'P03', 'F02', 'M04', 'P04'}


In [13]:
PC = list(PC_highCC)
PC.sort()
print(PC)
NC = list(NC_highCC)
NC.sort()
print(NC)

['A05', 'A06', 'A08', 'B05', 'B06', 'B07', 'C05', 'C06', 'C07', 'D05', 'D06', 'E05', 'E06', 'E08', 'F05', 'F06', 'G05', 'G06', 'G08', 'H05', 'H06', 'H07', 'H08', 'I05', 'I06', 'I07', 'I08', 'J05', 'J06', 'J07', 'K05', 'L05', 'L06', 'L07', 'M05', 'M07', 'N05', 'N06', 'N07', 'N08', 'O05', 'O06', 'O08', 'P05', 'P06']
['A01', 'A02', 'A03', 'B01', 'B02', 'B03', 'B04', 'C01', 'C02', 'C03', 'C04', 'D01', 'D02', 'D03', 'D04', 'E01', 'E02', 'E03', 'E04', 'F01', 'F02', 'F03', 'F04', 'G01', 'G04', 'H01', 'H03', 'H04', 'I04', 'J01', 'J02', 'K01', 'L03', 'L04', 'M01', 'M02', 'M04', 'N01', 'N02', 'N04', 'O01', 'O04', 'P01', 'P03', 'P04']


In [7]:
meta = meta.loc[meta['Image_Metadata_WellID'].isin(NC+PC)]
wells  = meta['ImageNumber'].unique().tolist()
meta.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,TimeSeries_20221028_164132
1,2,A01,TimeSeries_20221028_164132
2,3,A01,TimeSeries_20221028_164132
3,4,A01,TimeSeries_20221028_164132
4,5,A01,TimeSeries_20221028_164132


In [8]:
query ="SELECT * FROM MyExpt_Per_Object WHERE ImageNumber IN("

In [9]:
query+=f"{wells[0]}"
for w in wells[1:]:
    query+=","
    query+=f"{w}"
query+=")"

In [10]:
data = pd.read_sql_query(query, con)
data.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_EproAfterMath_6_02_256,Nuclei_Texture_Variance_EproAfterMath_6_03_256,Nuclei_Texture_Variance_Hoe_6_00_256,Nuclei_Texture_Variance_Hoe_6_01_256,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,0.0,0.0,9.712645,10.273859,9.904125,10.098906,0.0,0.0,0.0,0.0
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,0.0,0.0,6.579749,6.315129,5.832482,6.946683,0.0,0.0,0.0,0.0
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,0.0,0.0,15.497243,16.163633,15.251661,16.716883,0.0,0.0,0.0,0.0
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,0.0,0.0,13.691379,13.019686,13.500400,13.272403,0.0,0.0,0.0,0.0
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.0,0.0,0.673830,0.660640,0.689110,0.654166,0.0,0.0,0.0,0.0


In [39]:
df = pd.merge(data, meta, on='ImageNumber')
df['Image_Metadata_WellID'].unique()

array(['A01', 'A02', 'A03', 'A05', 'A06', 'A08', 'B01', 'B02', 'B03',
       'B04', 'B05', 'B06', 'B07', 'C01', 'C02', 'C03', 'C04', 'C05',
       'C06', 'C07', 'D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'E01',
       'E02', 'E03', 'E04', 'E05', 'E06', 'E08', 'F01', 'F02', 'F03',
       'F04', 'F05', 'F06', 'G01', 'G04', 'G05', 'G06', 'G08', 'H01',
       'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'I04', 'I05', 'I06',
       'I07', 'I08', 'J01', 'J02', 'J05', 'J06', 'J07', 'K01', 'K05',
       'L03', 'L04', 'L05', 'L06', 'L07', 'M01', 'M02', 'M04', 'M05',
       'M07', 'N01', 'N02', 'N04', 'N05', 'N06', 'N07', 'N08', 'O01',
       'O04', 'O05', 'O06', 'O08', 'P01', 'P03', 'P04', 'P05', 'P06'],
      dtype=object)

In [40]:
df.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_Hoe_6_00_256,Nuclei_Texture_Variance_Hoe_6_01_256,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,9.712645,10.273859,9.904125,10.098906,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,6.579749,6.315129,5.832482,6.946683,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,15.497243,16.163633,15.251661,16.716883,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,13.691379,13.019686,13.500400,13.272403,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.673830,0.660640,0.689110,0.654166,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132


In [41]:
df['con'] = 'NC'
df.loc[df['Image_Metadata_WellID'].isin(PC), 'con'] = 'PC'

In [42]:
df['label'] = 0
df.loc[df['con']=='PC', 'label'] = 1

In [43]:
df.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256,Image_Metadata_WellID,Image_Metadata_PlateID,con,label
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,9.904125,10.098906,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,5.832482,6.946683,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,15.251661,16.716883,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,13.500400,13.272403,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.689110,0.654166,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0


In [44]:
meta_cols = df.columns[df.columns.str.contains(pat='Metadata|ImageNumber|Location|Center|Cells_Number_Object_Number|ObjectNumber', flags=re.IGNORECASE)].tolist()


In [17]:
cols = df.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()

In [18]:
#FOR time series
data_cols = df[cols].columns[df[cols].columns.str.contains(pat='Epro|NS4B',flags=re.IGNORECASE)].tolist()

In [35]:
#FOR DR
with open('data_cols_reduced_DR', 'rb') as f:
    data_cols = pickle.load(f)

In [36]:
data_cols

['Nuclei_Intensity_MassDisplacement_NS4BAfterMath',
 'Nuclei_Texture_Contrast_EproAfterMath_6_03_256',
 'Cells_Intensity_MaxIntensityEdge_EproAfterMath',
 'Cells_Intensity_IntegratedIntensity_NS4BAfterMath',
 'Cells_Intensity_MedianIntensity_EproAfterMath',
 'Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256',
 'Nuclei_Texture_SumVariance_NS4BAfterMath_6_01_256',
 'Cytoplasm_Texture_Entropy_EproAfterMath_6_03_256',
 'Cytoplasm_Texture_DifferenceEntropy_EproAfterMath_6_00_256',
 'Nuclei_Texture_Entropy_NS4BAfterMath_6_03_256',
 'Cytoplasm_Texture_InfoMeas2_NS4BAfterMath_6_03_256',
 'Nuclei_Intensity_UpperQuartileIntensity_EproAfterMath',
 'Cytoplasm_Texture_SumEntropy_NS4BAfterMath_6_00_256',
 'Cytoplasm_Texture_DifferenceVariance_EproAfterMath_6_01_256',
 'Nuclei_Texture_InfoMeas1_EproAfterMath_6_00_256',
 'Cells_Intensity_UpperQuartileIntensity_EproAfterMath',
 'Nuclei_Texture_InfoMeas2_EproAfterMath_6_03_256',
 'Nuclei_Texture_Correlation_EproAfterMath_6_00_256',
 'Nuc

In [37]:
cols = ['ImageNumber', 'Image_Metadata_WellID', 'Image_Metadata_PlateID']+data_cols

In [45]:
df[data_cols] = StandardScaler().fit_transform(df[data_cols])

In [2]:
import xgboost
model = xgboost.XGBRegressor()

In [ ]:
#Harmony!

In [5]:
harmony = pd.read_sql_query('SELECT * from harmony_img_level', con)
harmony.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V14,V15,V16,V17,V18,V19,V20,Image_Metadata_WellID,Image_Metadata_PlateID,ImageNumber
0,0.305695,-0.510150,-0.167764,-0.367259,-0.038895,0.081975,0.029306,-0.179630,-0.086734,0.073944,...,-0.186315,-0.055696,0.028117,0.020501,0.115348,0.082253,-0.099590,A01,TimeSeries_20221028_164132,1
1,0.308491,-0.534324,-0.078357,-0.347420,0.015375,0.167542,0.065436,-0.146419,-0.066748,0.041076,...,-0.180165,-0.050452,-0.016776,0.002821,0.105768,0.069213,-0.050671,A01,TimeSeries_20221028_164132,2
2,0.229197,-0.630664,-0.010510,-0.350601,-0.134096,0.000786,0.196617,-0.031949,-0.030125,-0.136287,...,-0.193635,-0.038927,-0.039317,0.040172,0.120463,0.117115,0.027497,A01,TimeSeries_20221028_164132,3
3,0.219075,-0.603700,-0.136947,-0.363700,-0.188136,-0.085320,0.145746,-0.135266,-0.083496,-0.080876,...,-0.240516,-0.040474,-0.016773,0.027725,0.132825,0.081961,0.036667,A01,TimeSeries_20221028_164132,4
4,0.180284,-0.633803,-0.181292,-0.433847,-0.078915,0.095016,-0.024631,-0.103691,-0.116034,-0.081669,...,-0.282634,-0.013972,0.004544,0.093749,0.081393,0.077621,-0.061731,A01,TimeSeries_20221028_164132,5


In [ ]:
TS_only = harmony[harmony['Image_Metadata_PlateID'] == "TimeSeries_20221028_164132"]
TS_subset = TS_only.sample(frac=0.2, random_state=42)

In [44]:
harmony = harmony.drop(TS_subset.index)

In [7]:
PC_DR = ['A01', 'A02', 'B01', 'B02', 'C01', 'C02', 'D01', 'D02', 'E01', 'E02', 'F01', 'F02', 'G01', 'G02', 'H01', 'H02', 'I01', 'I02', 'J01', 'J02', 'K01', 'K02', 'L01', 'L02', 'M01', 'M02', 'N02', 'O01', 'O02', 'P01', 'P02'] 
NC_DR = ['A23', 'A24', 'B23', 'B24', 'C23', 'C24', 'D23', 'D24', 'E23', 'E24', 'F23', 'F24', 'G23', 'G24', 'H23', 'H24', 'I23', 'I24', 'J23', 'J24', 'K23', 'K24', 'L23', 'L24', 'M23', 'M24', 'N23', 'N24', 'O23', 'O24', 'P23', 'P24']

In [46]:
#select only control wells
#loc by plate ID and well ID for their respective controls
harm_TS = harmony.loc[(harmony['Image_Metadata_WellID'].isin(PC+NC)) & (harmony['Image_Metadata_PlateID'] == 'TimeSeries_20221028_164132')]
harm_DR = harmony.loc[(harmony['Image_Metadata_WellID'].isin(PC_DR+NC_DR)) & (harmony['Image_Metadata_PlateID'] == 'DR_20221209_143805')]
#apply PC and NC labels
harm_TS['con'] = 'NC'
harm_TS.loc[harm_TS['Image_Metadata_WellID'].isin(PC), 'con'] = 'PC'
harm_DR['con'] = 'NC'
harm_DR.loc[harm_DR['Image_Metadata_WellID'].isin(PC_DR), 'con'] = 'PC'
harm_TS['label'] = 0
harm_TS.loc[harm_TS['con']=='PC', 'label'] = 1
harm_DR['label'] = 0
harm_DR.loc[harm_DR['con']=='PC', 'label'] = 1

harmony_labeled = pd.concat([harm_TS, harm_DR], ignore_index = True)
harmony_labeled.head()

<ipython-input-46-226b7cf2446e>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  harm_TS['con'] = 'NC'
<ipython-input-46-226b7cf2446e>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  harm_DR['con'] = 'NC'
<ipython-input-46-226b7cf2446e>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-v

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V16,V17,V18,V19,V20,Image_Metadata_WellID,Image_Metadata_PlateID,ImageNumber,con,label
0,0.305695,-0.510150,-0.167764,-0.367259,-0.038895,0.081975,0.029306,-0.179630,-0.086734,0.073944,...,0.028117,0.020501,0.115348,0.082253,-0.099590,A01,TimeSeries_20221028_164132,1,NC,0
1,0.308491,-0.534324,-0.078357,-0.347420,0.015375,0.167542,0.065436,-0.146419,-0.066748,0.041076,...,-0.016776,0.002821,0.105768,0.069213,-0.050671,A01,TimeSeries_20221028_164132,2,NC,0
2,0.229197,-0.630664,-0.010510,-0.350601,-0.134096,0.000786,0.196617,-0.031949,-0.030125,-0.136287,...,-0.039317,0.040172,0.120463,0.117115,0.027497,A01,TimeSeries_20221028_164132,3,NC,0
3,0.219075,-0.603700,-0.136947,-0.363700,-0.188136,-0.085320,0.145746,-0.135266,-0.083496,-0.080876,...,-0.016773,0.027725,0.132825,0.081961,0.036667,A01,TimeSeries_20221028_164132,4,NC,0
4,0.180284,-0.633803,-0.181292,-0.433847,-0.078915,0.095016,-0.024631,-0.103691,-0.116034,-0.081669,...,0.004544,0.093749,0.081393,0.077621,-0.061731,A01,TimeSeries_20221028_164132,5,NC,0


In [17]:
meta_cols = harmony_labeled.columns[harmony_labeled.columns.str.contains(pat='Metadata|ImageNumber|Location|Center|Cells_Number_Object_Number|ObjectNumber', flags=re.IGNORECASE)].tolist()


In [18]:
cols = harmony_labeled.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()

In [23]:
from sklearn.model_selection import train_test_split

In [47]:
X = harmony_labeled[cols]
y = harmony_labeled['label']

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [49]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             n_estimators=100, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=None, ...)

In [27]:
import pickle

In [28]:
with open('data_cols_reduced', 'wb') as f:
    pickle.dump(data_cols, f)

In [50]:
preds = model.predict(X_test)

In [51]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y_true=y_test, y_pred=preds)
mse = mean_squared_error(y_true=y_test, y_pred=preds)

In [52]:
print("R2", r2)
print("MSE", mse)

R2 0.9353571762126521
MSE 0.01614009280149662


In [53]:
model.save_model('xgb_model_0_48_harmonized')

In [53]:
del df

In [54]:
scores = model.predict(TS_subset[cols])

In [55]:
TS_subset['score'] = scores
TS_subset.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V15,V16,V17,V18,V19,V20,Image_Metadata_WellID,Image_Metadata_PlateID,ImageNumber,score
1259,0.348318,0.630771,0.114394,-0.108108,-0.035819,0.032318,-0.067389,-0.220442,0.171624,0.008820,...,-0.027523,-0.047278,-0.008073,0.024795,-0.002990,-0.010499,K10,TimeSeries_20221028_164132,2221,1.013244
1623,0.236634,0.827524,0.282327,-0.078409,-0.117510,0.002932,-0.001621,-0.158959,0.062518,0.001878,...,0.015500,-0.050538,0.010074,-0.023968,0.002836,0.034145,N07,TimeSeries_20221028_164132,2832,1.000247
611,0.354396,-1.205397,1.404094,-0.675560,-0.218807,-0.425428,-0.175637,0.114739,0.082322,-0.087571,...,-0.365504,-0.158916,-0.074151,0.030523,0.021460,0.275030,E16,TimeSeries_20221028_164132,984,1.025692
514,0.611327,-0.326937,-0.213374,-0.038054,-0.040221,0.044905,0.122118,0.072502,-0.094285,-0.006367,...,0.013820,0.025345,0.029787,-0.104759,0.000501,0.079323,E01,TimeSeries_20221028_164132,850,-0.000635
413,0.707695,-0.275301,-0.179546,-0.097304,0.156718,0.058836,0.031449,-0.027455,-0.058045,0.003796,...,0.012700,0.034154,-0.026920,0.014317,0.043295,-0.016287,D04,TimeSeries_20221028_164132,656,-0.000279


In [56]:
TS_subset.to_csv("timeseries_harmonized_makegrafwiththis.csv")